# Machine_Learning

## Chargement libraires

In [22]:
import sys
from pathlib import Path
import os
from dotenv import load_dotenv
import logging
import time
import json
import pandas as pd

# Charge le .env en mémoire
load_dotenv()

# Ajouter la racine du projet au chemin de recherche Python
racine = Path("..").resolve()  # remonte d'un niveau si le notebook est dans notebooks/
sys.path.insert(0, str(racine))

# Maintenant tu peux importer n'importe quel module du projet
from config import RACINE

## Export des données à partir de PostgreSQL

In [23]:
import pandas as pd
import psycopg2
from pathlib import Path
from datetime import datetime
from config import POSTGRES_HOST, POSTGRES_PORT, POSTGRES_DB, \
                   POSTGRES_USER, POSTGRES_PASSWORD

#RACINE      = Path(__file__).parent.parent.parent
DATASET_DIR = RACINE / "data" / "ml" / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
TIMESTAMP   = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_PATH = DATASET_DIR / f"dataset_ml_csv_{TIMESTAMP}.csv"

def export_csv_dataset():
    conn = psycopg2.connect(
        host=POSTGRES_HOST, port=POSTGRES_PORT,
        dbname=POSTGRES_DB, user=POSTGRES_USER,
        password=POSTGRES_PASSWORD
    )

    # Jointure des 3 tables en une seule requête
    query = """
        SELECT
            o.id,
            o.source,
            o.titre,
            o.description,
            o.type_contrat,
            o.localisation_ville,
            o.salaire_min,
            o.salaire_max,
            o.experience_min,
            o.secteur,
            o.date_publication,
            ml.ml_keyword,
            ml.ml_label_binaire,
            ml.ml_label_categorie,
            STRING_AGG(DISTINCT c.competence, ', ') AS competences,
            STRING_AGG(DISTINCT m.mission,    ' | ') AS missions
        FROM offres o
        LEFT JOIN competences c ON c.offre_id = o.id
        LEFT JOIN missions    m ON m.offre_id = o.id
        LEFT JOIN ml_labels   ml ON ml.offre_id = o.id
        GROUP BY o.id, o.source, o.titre, o.description,
                o.type_contrat, o.localisation_ville,
                o.salaire_min, o.salaire_max, o.experience_min,
                o.secteur, o.date_publication, ml.ml_keyword,
                ml.ml_label_binaire, ml.ml_label_categorie
    """

    df = pd.read_sql(query, conn)
    df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
    print(f"Dataset exporté : {len(df)} offres")

    conn.close()

#export_csv_dataset()

## Chargement données

In [29]:
# Charger les offres normalisées produites
fichiers = sorted(
    (RACINE / "data" / "ml" / "dataset").glob("*.csv")
)

if not fichiers:
    print("Aucun fichier CSV trouvé")
else:
    df     = pd.read_csv(fichiers[-1], encoding="utf-8", low_memory=False)
    offres = df.to_dict(orient="records")

    print(f"{len(offres)} offres chargées depuis {fichiers[-1].name}")

49750 offres chargées depuis dataset_ml_csv_20260423_074301.csv


## Exploration données

### Vérification des doublons

In [31]:
df.duplicated().sum() 

np.int64(0)

In [32]:
print(f"Il n'y a pas de doublons dans {fichiers[-1].name}")

Il n'y a pas de doublons dans dataset_ml_csv_20260423_074301.csv


### Données manquantes

In [33]:
# Décompte du nombre de données manquantes (NaN) pour chaque colonne
df.isnull().sum().sort_values(ascending=False) 

competences           48716
salaire_max           42098
salaire_min           42060
experience_min        23829
missions              11978
secteur                8442
id                        0
source                    0
type_contrat              0
localisation_ville        0
titre                     0
description               0
ml_keyword                0
date_publication          0
ml_label_categorie        0
ml_label_binaire          0
dtype: int64

In [35]:
# Pourcentage de données manquantes (NaN) pour chaque colonne
df.isnull().sum().sort_values(ascending=False)/len(df) *100

competences           97.921608
salaire_max           84.619095
salaire_min           84.542714
experience_min        47.897487
missions              24.076382
secteur               16.968844
id                     0.000000
source                 0.000000
type_contrat           0.000000
localisation_ville     0.000000
titre                  0.000000
description            0.000000
ml_keyword             0.000000
date_publication       0.000000
ml_label_categorie     0.000000
ml_label_binaire       0.000000
dtype: float64

In [40]:
# Aperçu de quelques colonnes
cols = ['source', 'titre', 'type_contrat',
       'localisation_ville', 'salaire_min', 'salaire_max', 'experience_min',
       'secteur', 'date_publication', 'ml_keyword', 'ml_label_binaire',
       'ml_label_categorie']

for column in cols:
    print('*'*30)
    print(f"Colonne étudiée : {column}")
    print('*'*30)
    print(df[column].value_counts())
    print('\n')

******************************
Colonne étudiée : source
******************************
source
welcometothejungle    38000
francetravail         11750
Name: count, dtype: int64


******************************
Colonne étudiée : titre
******************************
titre
Data Engineer                                                   798
Business analyst                                                713
Développeur / Développeuse full-stack                           479
Data Engineer (H/F)                                             342
Data analyst                                                    332
                                                               ... 
Manager support technique                                         1
Planneur / Planneuse senior                                       1
Courtier / Courtière en gestion de patrimoine                     1
Technicien / Technicienne de développement industrie méthode      1
Développeur / Développeuse IOT                    